<a href="https://colab.research.google.com/github/shardeep-x/agentic-ai-and-gen-ai/blob/foundation_of_gen_ai_llm/Aug_1_%5BC2%5D%5Blec_2%5D_Prompt_optimization_and_compression.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Prompt Optimisation and Compression

## Contents
1. Prompt Compression using LLMLingua
2. Prompt Optimisation

## Prompt Compression

Using LLMLingua2

**Step 0**: Install and import the required libraries (`transformers`, `datasets`, `llmlingua`, `torch`, `pandas`)

**Step 1**: Load the text generation model (Llama 3.2) and tokenizer

Model:
- unsloth/Llama-3.2-3B-Instruct



In [1]:
!pip -q install llmlingua datasets rouge_score

  Preparing metadata (setup.py) ... done


In [2]:
# import the libraries

In [3]:
# load the model using pipeline


In [4]:
# save the tokenzer for later use

**Step 2**: Define helper functions for text generation and token counting

- `generate_answer()`
- `count_tokens()`



    [{"generated_text":                              
        [                                            
            {"role": "system", "content": "..."},    
            {"role": "user", "content": "..."},      
            {"role": "assistant", "content": "..."}  
        ]                                            
    }]

In [5]:
def generate_answer(prompt):

    messages = [
        {
            "role": "user",
            "content": prompt
        }
    ]

    output = llm(messages)

    return output[0]["generated_text"][-1]["content"]

In [6]:
# define the count_tokens function


**Step 3**: Load the business earnings call dataset

Dataset:
- Aiera/aiera-ect-sum



#### An earnings call transcript is a real conversation between:

Company executives (CEO, CFO, etc.),Financial analysts,Investors

These transcripts are often very long, making them ideal for demonstrating prompt compression.

The dataset contains:

Earnings call transcripts
Human-written summaries

It was created for summarization research.Each example typically contains:

1.transcript

2.summary

In [7]:
# load the dataset


In [8]:
# print one row of the dataset


**Step 4**: Select an earnings call transcript for the demonstration

- Choose a transcript of suitable length
- Extract the transcript as the context



In [9]:
# extract the transcript of  any instance of the dataset


In [10]:
# print the number of characters of the transcript using the len() function


**Step 5**: Create the prompt

- Define the summarization instruction
- Combine the transcript and the instruction to form the original prompt
- Count the original prompt tokens



In [11]:
question = """
Summarize the earnings call in three bullet points.

1. Revenue reported.
2. Key business drivers.
3. Future guidance.
"""




In [12]:
# create the original prompt using the transcript and question

In [13]:
# print the number of tokens of the original prompt


**Step 6:** Generate a summary using the original prompt

- Pass the original prompt to Llama.
- Display the generated summary.

In [14]:
# generate an initial answer to the original prompt using the generate_answer function


In [15]:
# print the initial answer to the original Prompt


**Step 7**: Load the LLMLingua prompt compression model

Model:
- microsoft/llmlingua-2-xlm-roberta-large-meetingbank



In [16]:
# load the model using PromptCompressor class


**Step 8**: Compress the transcript using LLMLingua

- Compress the context
- Create the compressed prompt by appending the original instruction



In [17]:
# Compress the transcript


In [18]:
# show what the compress_prompt method returns


In [19]:
#create the compressed_prompt


**Step 9**: Compare the original and compressed prompts

- Count the original tokens
- Count the compressed tokens
- Calculate the compression percentage



In [20]:
# count the tokens of both the original and the compressed prompts


In [21]:
print(
    "Reduction         :",
    f"{(1-compressed_tokens/original_tokens)*100:.2f}%"
)

NameError: name 'compressed_tokens' is not defined

**Step 10**: Generate a summary using the compressed prompt

- Pass the compressed prompt to Llama
- Display the generated summary



In [ ]:
# generate a compressed_answer of the compressed prompt using the generatae_answer function



In [ ]:
# print the compressed_answer to the compressed prompt


**Step 11**: Compare the outputs

- Original prompt length
- Compressed prompt length
- Compression ratio
- Original summary
- Compressed summary
- Observe whether prompt compression preserves the important information while reducing the number of tokens.

In [ ]:
# extarct the ground truth summary


In [ ]:
# print the ground truth summary

In [ ]:
from rouge_score import rouge_scorer
scorer = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)

# Original answer compared with itself (reference)
original_rouge = scorer.score(initial_answer,human_summary)["rougeL"].fmeasure

# Compressed answer compared with original answer
compressed_rouge = scorer.score(compressed_answer, human_summary)["rougeL"].fmeasure
comparison = pd.DataFrame({

    "Method": [
        "Original Prompt",
        "LLMLingua"
    ],

    "Prompt Tokens": [
        original_tokens,
        compressed_tokens
    ],

    "Generated Answer": [
        initial_answer,
        compressed_answer
    ],

    "ROUGE-L Score": [
        round(original_rouge, 4),
        round(compressed_rouge, 4)
    ]

})

comparison

## Loading LLM and defining utility functions

In [ ]:
# load the model if not already done


    [{"generated_text":                              
        [                                            
            {"role": "system", "content": "..."},    
            {"role": "user", "content": "..."},      
            {"role": "assistant", "content": "..."}  
        ]                                            
    }]                                               

In [ ]:
# ============================================================
# Utility Functions
# ============================================================

def make_message(system_content, user_content):
    return [
        {"role": "system", "content": system_content},
        {"role": "user", "content": user_content},
    ]


def fetch_response(response):
    return response[0]["generated_text"][-1]["content"]



In [ ]:

# define the generate function to generate a response from the model given a system prompt and a user prompt


## Prompt Optmization

We will use the **ProTeGi** automatic prompt optimization algorithm for this demonstration. ProTeGi uses textual gradients to find flaws in the initial prompt and uses another (or same) LLM to create optimized version of the prompt.

**Scenario**: An application for a loan has been received by the bank. They have to decide whether to accept or reject the loan based upon certain lending criteria and parameters about the applicant.

It is difficult to manually write a prompt to automate this process. It is also difficult to manually optimize a simple prompt for such a complex scenario. Automatic prompt optimization is useful in this case to derive an optimized prompt starting from a simple prompt.

### ProTeGi: Gradient-Based Prompt Optimization Workflow

#### **Step 0:** Background info


##### Credit Policy

In [ ]:
credit_policy = """
NORTHBRIDGE BANK - RETAIL LENDING POLICY

P1. Debt-to-Income Ratio (DTI) = Existing Debt / Annual Income.
    DTI must be less than 0.40.

P2. Loan-to-Income Ratio (LTI) = Requested Loan / Annual Income.
    LTI must not exceed 3.0.

P3. Minimum credit score is 660.
    Scores between 620 and 659 are acceptable ONLY if collateral is provided.

P4. Applicants must have at least 12 months of continuous employment.

P5. Unsecured loans above $100,000 are not permitted.

P6. Final decision must be one of:
    APPROVE
    REJECT
    CONDITIONAL APPROVAL
"""


##### Loan Application

In [ ]:
loan_application = """
Applicant Name: John Smith

Age: 28

Annual Income: $48,000

Employment:
Software Developer at ABC Technologies

Employment Duration:
10 months

Credit Score:
615

Existing Debt:
$22,000

Requested Loan Amount:
$180,000

Loan Purpose:
Purchase a house

Collateral:
None

Previous Loan Defaults:
None
"""


##### Ground Truth

For this applicant,\
DTI = 22000 / 48000\
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;= 0.458 (violate P1)

LTI = 180000 / 48000\
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;= 3.75 (violate P2)

Applicant Score is 615 with no collateral (violate P3)

Employment Duration is 10 months (violate P4)

Applicant request of loan 180000 with no collateral (violate p5)

**Correct decision is reject**

In [ ]:
correct_decision = "reject"

#### **Step 1:** Define the Initial Prompt

##### The initial prompt

In [ ]:
system_prompt = """
You are a bank loan officer.

Review the loan application and decide whether the loan should be approved.

Provide a brief explanation.
"""

user_prompt = f"""
BANK LENDING POLICY

{credit_policy}

----------------------------------------

LOAN APPLICATION

{loan_application}
"""


#### **Step 2:** Generate the Initial Response

In [ ]:
#  generate the inital answer

In [ ]:
# print the initial answer


##### **Analysis of the Initial Response**



The final decision is not consistent with the bank's lending policy. The following issues can be observed:

Mistakes in the Model's Response
- Incorrect Final Decision
- Incorrect Credit Score Interpretation

However, after identifying these policy violations, it still concludes with Conditional Approval, making the reasoning internally inconsistent.

#### **Step 3:** Evaluate the Response
- Use an **LLM-as-a-Judge** (can be same or different model) to evaluate the generated response.
- The judge assigns a score and identifies:
  - Correct reasoning
  - Missing information
  - Errors or hallucinations
  - Policy violations

##### LLM Judge Prompt

In [ ]:
judge_system_prompt = """
You are an expert loan approval auditor.

Evaluate ONLY the quality of the model's reasoning.
Use the following rubric (10 points).
1. Correctly evaluates the credit score. (1 point)
2. Correctly evaluates annual income. (1 point)
3. Correctly evaluates existing debt. (1 point)
4. Correctly evaluates employment stability. (1 point)
5. Correctly evaluates collateral. (1 point)
6. Computes or discusses the Debt-to-Income (DTI) ratio. (1 point)
7. Computes or discusses the Loan-to-Income (LTI) ratio. (1 point)
8. References the lending policy when making the decision. (1 point)
9. Gives a clear evidence-based justification. (1 point)
10. Correct final decision. (1 point)

Be STRICT.
Only award a point if the criterion is explicitly satisfied.
A generic explanation should score between 3 and 5.
A detailed policy-based analysis should score between 8 and 10.

Return EXACTLY in this format:

Score: X/10
Strengths:
- ...
Missing:
- ...
"""

judge_user_prompt = f"""
BANK POLICY

{credit_policy}

----------------------------------------

LOAN APPLICATION

{loan_application}

----------------------------------------

MODEL RESPONSE

{initial_answer}

----------------------------------------

ACTUAL FINAL DECISION

{correct_decision}
"""

In [ ]:
# generate judge feedback


In [ ]:
# print Judge Feedback


### **Step 4:** Generate a Textual Gradient
Generate a **textual gradient**, i.e., natural-language feedback describing how the prompt should be improved.

##### Gradient Prompt

In [ ]:
gradient_system_prompt = """
Your task is to find out the textual gradients of a prompt. The textual gradient is the analysis of why the CURRENT PROMPT produced a weak response.

Do NOT rewrite the prompt.

Instead, identify the instructions that are missing from the prompt.

Focus on:

- Missing reasoning steps
- Missing financial analyses
- Missing policy references
- Missing output structure
- Missing justification requirements

Write 4-6 concise bullet points.

Each bullet should explain:

- what the prompt failed to instruct
- why that caused a weaker answer

Return ONLY the textual gradient.
"""

gradient_user_prompt = f"""
CURRENT PROMPT

{system_prompt}

--------------------------------------------------

BANK POLICY

{credit_policy}

--------------------------------------------------

LOAN APPLICATION

{loan_application}

--------------------------------------------------

MODEL RESPONSE

{initial_answer}

--------------------------------------------------

JUDGE FEEDBACK

{judge_feedback}

Generate the textual gradient.
"""

In [ ]:
# Generate Textual Gradient


In [ ]:
# Print the textual gradient


### **Step 5:** Optimize the Prompt

##### Optimizer Prompt

In [ ]:
optimizer_system_prompt = """
You are an expert Prompt Engineer. Your task is to improve the prompt using ONLY the textual gradient.

Requirements:

- Preserve the original task.
- Incorporate the feedback from the textual gradient.
- Add only the missing instructions.
- Keep the prompt concise and professional.
- Do not include explanations or reasoning outside the prompt.
- Return ONLY the improved prompt.
"""

optimizer_user_prompt = f"""
ORIGINAL PROMPT

{system_prompt}

--------------------------------------------------

TEXTUAL GRADIENT

{textual_gradient}

--------------------------------------------------

Rewrite the prompt by incorporating the missing instructions identified in the textual gradient.
"""

In [ ]:
# generate the optimised prompt


In [ ]:
# print the optimised prompt


Analysis of the Optimized Prompt

Compared to the original prompt, the optimized prompt incorporates several important instructions identified through the textual gradient:

- Explicitly requires financial calculations
- Grounds the decision in the bank policy
- Improves justification quality
- Provides clearer decision instructions

### **Step 6:** Generate the Optimized Response
Generate a new response using the optimized prompt for the same input.



In [ ]:
# generate the optimised answer


In [ ]:
# print the optimised answer


### **Step 7:** Re-evaluate the Optimized Response
- Evaluate the optimized response using the same judge and evaluation criteria.
- Assign a new score to measure the improvement.



##### optimised judge user prompt

In [ ]:
optimized_judge_user_prompt = f"""
BANK POLICY

{credit_policy}

----------------------------------------

LOAN APPLICATION

{loan_application}

----------------------------------------

MODEL RESPONSE

{optimised_answer}

----------------------------------------

ACTUAL FINAL DECISION

{correct_decision}
"""

In [ ]:
# generate the optimised judge feedback


In [ ]:
# print the optimised judge feedback


### Analysis of the initial prompt

The initial prompt doesn't tell the model to:

  - calculate DTI, LTI
  - not justify policy violations
  - verify every policy rule
  - explain which rules passed or failed
  - justify the decision with evidence
  - use the allowed output labels exactly

The prompt is too generic.